# Cboe 总期权成交活跃度与 SOXX 未来五日波动

固定假设、来源、时钟、分窗、预算和信息门见 `precommitted_information_plan.json`。执行先验证第三次同日 Cboe 盘前版本与来源 Recorder、Qlib 文件身份；未通过不读取市场结果。历史版本仍不具备首次公开时间认证。


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import os, json, hashlib
import numpy as np
import pandas as pd
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
root=Path.cwd().resolve()
while not (root/"SPEC.md").is_file():
    if root.parent==root: raise RuntimeError("workspace root missing")
    root=root.parent
card=root/"research/cboe_total_activity_variance_information_20261022"
source_card=root/"research/cboe_total_options_activity_source_20261022"
plan_path=card/"precommitted_information_plan.json"
binding_path=card/"execution_binding.json"
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
plan=json.loads(plan_path.read_text())
binding=json.loads(binding_path.read_text())
assert plan["frozen"] and plan["execution_authorized"] and binding["plan_sha256"]==sha(plan_path)
assert not (card/"run_state.json").exists() and not (card/"summary.json").exists(),"fixed attempt already started"
for name,key in [("summary.json","summary_sha256"),("source_coverage_audit.json","coverage_audit_sha256"),
                 ("source_scale_diagnostic.json","source_scale_diagnostic_sha256"),
                 ("publication_window.json","publication_window_sha256"),
                 ("revision_check_20261022.json","same_day_revision_check_sha256")]:
    assert sha(source_card/name)==plan["source"][key],name
for relative,digest in binding["provider_files_sha256"].items():
    assert sha(root/relative)==digest,relative
assert sha(source_card/"verification_manifest.json")==binding["source_verification_manifest_sha256"]
assert sha(source_card/"forward_publication_plan.json")==binding["forward_plan_sha256"]
assert sha(source_card/"forward_transport_addendum.json")==binding["forward_addendum_sha256"]
receipts=sorted((source_card/"forward_observations").glob("*.json"))
assert len(receipts)==3,"third source observation is required exactly once"
for name,digest in binding["prior_receipts_sha256"].items():
    assert sha(source_card/"forward_observations"/name)==digest,name
third=json.loads(receipts[-1].read_text())
second=json.loads((source_card/"forward_observations"/sorted(binding["prior_receipts_sha256"])[-1]).read_text())
assert third["index"]==3 and third["status"]=="observed" and third["plan_sha256"]==binding["forward_plan_sha256"]
assert {x["symbol"] for x in third["records"]}=={"SOXX","SOXL"}
second_by_symbol={x["symbol"]:x for x in second["records"]}
for item in third["records"]:
    assert datetime.fromisoformat(item["requested_at_utc"])>=datetime.fromisoformat(binding["third_receipt_not_before_utc"])
    prior=second_by_symbol[item["symbol"]]
    assert item["http_status"]==200 and item["target_date_present"] and item["earlier_date_unchanged"]
    assert item["url"]==prior["url"] and item["sha256"]==prior["sha256"]
    assert item["daily_contracts"]==prior["daily_contracts"]
    assert sha(root/item["relative_path"])==item["sha256"]
# All identity and publication checks finish before any SOXX price or outcome read.
import qlib
from qlib.data import D
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from src.core.recorder_sources import load_verified_recorder_objects
from src.workflow.inventory_sales_soxx_h63_information import stationary_bootstrap_lower
qlib.init(provider_uri=str(root/plan["provider"]["uri"]),region="us",kernels=1)
objects,source_audit=load_verified_recorder_objects(source_card,
    expected_recorder_id=binding["source_recorder_id"],artifacts=[binding["source_artifact"]],
    expected_artifact_sha256={binding["source_artifact"]:binding["source_artifact_sha256"]},
    expected_manifest_sha256=binding["source_verification_manifest_sha256"])
observed=objects[binding["source_artifact"]]
assert set(observed)=={"SOXX","SOXL"}
source=pd.Series(observed["SOXX"]["daily_total_contracts"]).sort_index().astype(float)
source.index=pd.DatetimeIndex(source.index).normalize()
assert source.index.is_unique and source.index.is_monotonic_increasing
assert source.index[0]==pd.Timestamp(plan["provider"]["source_count_bounds"][0])
assert source.index[-1]==pd.Timestamp(plan["provider"]["source_count_bounds"][1])
assert source.gt(0).all()
cal=pd.DatetimeIndex(D.calendar(start_time=plan["provider"]["calendar_bounds"][0],
    end_time=plan["provider"]["calendar_bounds"][1],freq="day")).normalize()
assert cal.is_unique and cal.is_monotonic_increasing
assert source.index.equals(cal[:-1]),"Cboe source dates must match every Qlib day except final day"
raw=D.features(["SOXX"],plan["provider"]["fields"],start_time=cal[0],end_time=cal[-1],freq="day").reset_index()
assert set(raw.instrument)=={"SOXX"}
raw["datetime"]=pd.to_datetime(raw.datetime).dt.normalize()
assert pd.DatetimeIndex(raw.datetime).equals(cal)
assert raw[["$open","$close","$volume"]].notna().all().all()
assert raw[["$open","$close","$volume"]].gt(0).all().all()
prices=raw.set_index("datetime")[["$open","$close","$volume"]].astype(float)
close,opening,volume=prices["$close"],prices["$open"],prices["$volume"]
ret=close.pct_change(fill_method=None)
frame=pd.DataFrame(index=cal)
frame["r1_lag"]=ret.shift(1)
frame["r5_lag"]=close.shift(1)/close.shift(6)-1
frame["rv5_lag"]=ret.shift(1).rolling(5,min_periods=5).std(ddof=0)
frame["rv22_lag"]=ret.shift(1).rolling(22,min_periods=22).std(ddof=0)
frame["volume_surprise_lag"]=np.log(volume.shift(1)/volume.shift(2).rolling(22,min_periods=22).median())
log_activity=np.log1p(source.reindex(cal))
frame["activity_surprise_lag"]=log_activity.shift(1)-log_activity.shift(2).rolling(22,min_periods=22).median()
daily=opening.shift(-2)/opening.shift(-1)-1
forward=pd.concat([daily.shift(-j) for j in range(5)],axis=1)
frame["log_rv5_label"]=np.log(1e-8+forward.pow(2).sum(axis=1))
frame.loc[forward.isna().any(axis=1),"log_rv5_label"]=np.nan
frame["mature_at"]=pd.Series(cal,index=cal).shift(-6)+pd.Timedelta("9h31min")
train=frame.loc[plan["windows"]["train_signal"][0]:plan["windows"]["train_signal"][1]].copy()
held=frame.loc[plan["windows"]["held_reused_diagnostic"][0]:plan["windows"]["held_reused_diagnostic"][1]].copy()
baseline=["r1_lag","r5_lag","rv5_lag","rv22_lag","volume_surprise_lag"]
candidate=baseline+["activity_surprise_lag"]
assert len(plan["feature_formula"]["baseline"])==len(baseline)
assert len(train)>=plan["windows"]["minimum_train_rows"] and len(held)>=plan["windows"]["minimum_held_rows"]
assert train[candidate+["log_rv5_label","mature_at"]].notna().all().all()
assert held[candidate+["log_rv5_label","mature_at"]].notna().all().all()
assert train.mature_at.max()<held.index.min()
assert held.mature_at.max()<=cal[-1]+pd.Timedelta("9h31min")
native=frame.loc[train.index.union(held.index),candidate+["log_rv5_label"]].copy()
native["instrument"]="SOXX";native.index.name="datetime"
native=native.reset_index().set_index(["datetime","instrument"])
panel=pd.concat({"feature":native[candidate],"label":native[["log_rv5_label"]]},axis=1)
dataset=DatasetH(handler=DataHandlerLP.from_df(panel),segments={"all":(None,None)})
dataset.config(dump_all=True,recursive=True)
prepared=dataset.prepare("all",col_set=["feature","label"])
pd.testing.assert_frame_equal(prepared,panel)
train_index=pd.MultiIndex.from_product([train.index,["SOXX"]],names=["datetime","instrument"])
held_index=pd.MultiIndex.from_product([held.index,["SOXX"]],names=["datetime","instrument"])
y_train=prepared["label"].loc[train_index,"log_rv5_label"].to_numpy(float)
y_held=prepared["label"].loc[held_index,"log_rv5_label"].to_numpy(float)
assert np.isfinite(y_train).all() and np.isfinite(y_held).all()
(card/"run_state.json").write_text(json.dumps({"status":"preflight_complete","plan_sha256":sha(plan_path),
    "binding_sha256":sha(binding_path),"source_loads":1,"D_calendar_calls":1,"D_features_calls":1,"DatasetH_builds":1})+"\n")
tracking=card/"qlib_recorder"
with R.start(experiment_name=plan["experiment"],uri=str(tracking)):
    recorder=R.get_recorder()
    (card/"run_state.json").write_text(json.dumps({"status":"fits_started","plan_sha256":sha(plan_path),
        "binding_sha256":sha(binding_path),"source_loads":1,"D_calendar_calls":1,"D_features_calls":1,"DatasetH_builds":1})+"\n")
    models={};prediction={}
    for name,features in (("baseline",baseline),("candidate",candidate)):
        scaler=StandardScaler()
        x_train=scaler.fit_transform(prepared["feature"].loc[train_index,features].to_numpy(float))
        x_held=scaler.transform(prepared["feature"].loc[held_index,features].to_numpy(float))
        assert np.isfinite(x_train).all() and np.isfinite(x_held).all() and (scaler.var_>0).all()
        model=Ridge(alpha=1.0)
        model.fit(x_train,y_train)
        models[name]={"scaler":scaler,"model":model,"features":features}
        prediction[name]=model.predict(x_held)
    held_result=held[["log_rv5_label","activity_surprise_lag","mature_at"]].copy()
    held_result["baseline_prediction"]=prediction["baseline"]
    held_result["candidate_prediction"]=prediction["candidate"]
    held_result["squared_error_improvement"]=(y_held-prediction["baseline"])**2-(y_held-prediction["candidate"])**2
    assert np.isfinite(held_result.squared_error_improvement).all()
    boot=plan["information_gate"]["stationary_bootstrap"]
    lower=stationary_bootstrap_lower(held_result.squared_error_improvement.to_numpy(),
        mean_block=boot["mean_block_sessions"],draws=boot["draws"],seed=boot["seed"],quantile=boot["lower_quantile"])
    half_means=[]
    for start,end in plan["windows"]["held_halves"]:
        values=held_result.loc[start:end,"squared_error_improvement"]
        assert len(values)>0
        half_means.append(float(values.mean()))
    coefficient=float(models["candidate"]["model"].coef_[-1])
    passed=bool(coefficient>0 and held_result.squared_error_improvement.mean()>0
                and all(value>0 for value in half_means) and lower>0)
    result={"status":"not_promoted_source_PIT_gap" if passed else "not_promoted_information_gate_failed",
        "train_rows":len(train),"held_rows":len(held),"candidate_activity_coefficient":coefficient,
        "held_mse_baseline":float(np.mean((y_held-prediction["baseline"])**2)),
        "held_mse_candidate":float(np.mean((y_held-prediction["candidate"])**2)),
        "held_paired_mse_improvement":float(held_result.squared_error_improvement.mean()),
        "held_half_improvements":half_means,"stationary_bootstrap_lower":lower,"information_gate_pass":passed,
        "historical_first_public_time_qualified":False,"historical_window_independent":False,
        "new_model_fits":2,"new_bootstraps":1,"new_native_accounts":0,"qualified_forward_days":0,
        "SOXX_goal_achieved":False,"trading_or_promotion_permission":False,
        "plan_sha256":sha(plan_path),"binding_sha256":sha(binding_path),
        "third_receipt_sha256":sha(receipts[-1]),"source_recorder_id":binding["source_recorder_id"]}
    recorder.save_objects(artifact_path="inputs",plan=plan,binding=binding,source_audit=source_audit)
    recorder.save_objects(artifact_path="sources",source_daily=source)
    recorder.save_objects(artifact_path="models",baseline=models["baseline"],candidate=models["candidate"])
    recorder.save_objects(artifact_path="predictions",held=held_result,train_features=train[candidate+["log_rv5_label"]])
    recorder.save_objects(artifact_path="analysis",result=result)
    recorder.log_metrics(train_rows=len(train),held_rows=len(held),
        held_mse_improvement=result["held_paired_mse_improvement"],bootstrap_lower=lower,
        information_gate_pass=int(passed),model_fits=2,native_accounts=0)
    result.update(recorder_id=recorder.id,experiment_id=str(recorder.experiment_id),recorder_uri=str(tracking),
        recorder_local_dir=str(tracking/str(recorder.experiment_id)/recorder.id))
artifacts=Path(result["recorder_local_dir"])/"artifacts"
verification={"recorder_id":result["recorder_id"],"entries":[{"path":str(path.relative_to(card)),
    "sha256":sha(path),"bytes":path.stat().st_size} for path in sorted(artifacts.rglob("*")) if path.is_file()]}
(card/"verification_manifest.json").write_text(json.dumps(verification,indent=2)+"\n")
(card/"result_review.json").write_text(json.dumps(result,indent=2)+"\n")
summary={"experiment":plan["experiment"],"status":result["status"],"decision":result["status"],
    "result":"result_review.json","recorder_id":result["recorder_id"],"experiment_id":result["experiment_id"],
    "recorder_uri":result["recorder_uri"],"recorder_local_dir":result["recorder_local_dir"],
    "model_fits":2,"native_accounts":0,"qualified_forward_days":0,"objective_achieved":False,
    "artifact_policy":plan["artifact_policy"]}
(card/"summary.json").write_text(json.dumps(summary,indent=2)+"\n")
(card/"conclusion.md").write_text("# 中文结论\n\n固定 Cboe 总期权成交量异常相对 SOXX 价格、已实现波动和股票成交量基线的五日未来波动信息筛选：2019–2022 训练 "+str(len(train))+" 日，2023–2026 已复用诊断 "+str(len(held))+" 日；两次 Ridge 拟合、一次区块 bootstrap、零账户。候选系数 "+f"{coefficient:.6f}"+"，持有期配对均方误差改善 "+f"{result['held_paired_mse_improvement']:.8f}"+"，下界 "+f"{lower:.8f}"+"；两个预登记持有阶段分别 "+f"{half_means[0]:.8f}"+" 和 "+f"{half_means[1]:.8f}"+"。信息门 "+("通过" if passed else "失败")+"。Recorder `"+result["recorder_id"]+"`。\n\nCboe 历史首次发布时间与版本未验证，复用市场历史不是独立前向；本卡不获晋升，没有 SOXX 费用后账户或独立前向天数。固定失败版本不改特征、符号或窗口；信息通过也只触发另行冻结经济账户设计。\n")
(card/"run_state.json").write_text(json.dumps({"status":"completed","plan_sha256":sha(plan_path),
    "binding_sha256":sha(binding_path),"recorder_id":result["recorder_id"]})+"\n")
print(json.dumps({"status":result["status"],"train":len(train),"held":len(held),
    "paired_improvement":result["held_paired_mse_improvement"],"lower":lower,
    "coefficient":coefficient,"recorder_id":result["recorder_id"]}))


## 中文结论

执行后以同卡 `conclusion.md`、`result_review.json`、`summary.json` 和原生 Qlib Recorder 为准；执行前无因子信息结果、账户或晋升结论。
